### Phase 1: Environment Setup & Dual-Model Loading
In this step, we load TWO distinct models:
1. **Base NLP Model:** Used for basic linguistic tasks (sentence splitting, tokenization).
2. **Custom NER Model:** The machine learning model you trained in `ner_train.ipynb`. This model has learned the *semantic patterns* of hard skills and can extract out-of-vocabulary (OOV) technologies.

In [6]:
import spacy
import time
from dataclasses import dataclass
from typing import List, Set, Dict

# 1. Load Base Model
try:
    nlp_base = spacy.load("en_core_web_sm")
except OSError:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp_base = spacy.load("en_core_web_sm")

# 2. Load Custom Trained NER Model
try:
    # If saved as a spaCy model directory:
    nlp_custom_ner = spacy.load("hard_skill_model.pkl")
    print("✅ Custom ML-NER Model loaded successfully!")
except OSError:
    print("⚠️ Custom model not found at specified path. Using base model as a placeholder for demonstration.")
    nlp_custom_ner = nlp_base # Fallback strictly for code execution

print("✅ Phase 1 Complete. Models are ready.")

⚠️ Custom model not found at specified path. Using base model as a placeholder for demonstration.
✅ Phase 1 Complete. Models are ready.


### Phase 2: True Hybrid Skill Matcher Engine
This class implements the Dual-Engine logic:
* **`_extract_via_era()`**: Uses the predefined dictionary (high precision, low recall).
* **`_extract_via_ner()`**: Uses the custom Machine Learning model to discover new skills (high recall, handles unseen data).
* **`merge_and_score()`**: Fuses the results, removes duplicates, and applies the ERA Priority Score formula ($Relevance \times Difficulty$).

In [7]:
@dataclass
class SkillReport:
    skill_name: str
    source: str         # Indicates if it was found by 'ERA_Dict' or 'ML_NER'
    required_level: str
    priority_score: float
    is_matched: bool

class TrueHybridMatcher:
    def __init__(self, nlp_base, nlp_ner):
        self.nlp_base = nlp_base
        self.nlp_ner = nlp_ner
        
        # --- ERA Engine Configurations ---
        self.era_taxonomy = {
            "Tableau": {"base_difficulty": 3},
            "Excel": {"base_difficulty": 2},
            "Data Analytics": {"base_difficulty": 3},
            "Reporting": {"base_difficulty": 2},
            "Python": {"base_difficulty": 3},
            "SQL": {"base_difficulty": 3}
        }
        self.importance_signals = ["require", "essential", "must", "optimize"]
        
        # --- NER Engine Configurations ---
        self.difficulty_map = {"Basic": 1, "Intermediate": 3, "Senior": 4, "Expert": 5}
        self.context_signals = {"expert": "Expert", "advanced": "Expert", "proficient": "Intermediate"}

    def _get_dynamic_difficulty(self, text: str, skill: str) -> str:
        """Uses base NLP to scan the sentence for difficulty modifiers."""
        doc = self.nlp_base(text.lower())
        for sent in doc.sents:
            if skill.lower() in sent.text:
                for token in sent:
                    if token.text in self.context_signals:
                        return self.context_signals[token.text]
        return "Intermediate" # Default if no signal is found

    def process_job_and_resume(self, jd_text: str, resume_text: str) -> List[SkillReport]:
        jd_lower = jd_text.lower()
        resume_lower = resume_text.lower()
        extracted_skills = {} # Use dict to deduplicate by skill name

        # --- ENGINE 1: ERA Dictionary Extraction ---
        for skill, info in self.era_taxonomy.items():
            if skill.lower() in jd_lower:
                extracted_skills[skill.lower()] = {
                    "name": skill,
                    "source": "ERA_Dict",
                    "base_diff": info["base_difficulty"]
                }

        # --- ENGINE 2: ML-NER Extraction ---
        # THIS is where your trained model shines! It reads the text and predicts entities.
        doc_ner = self.nlp_ner(jd_text)
        for ent in doc_ner.ents:
            # Assuming your custom model labels hard skills as 'HARD_SKILL' or similar
            # For demonstration, we'll accept any entity if using the fallback model
            skill_name = ent.text.strip().title()
            skill_key = skill_name.lower()
            
            # If the ML model found something the Dictionary missed!
            if skill_key not in extracted_skills:
                extracted_skills[skill_key] = {
                    "name": skill_name,
                    "source": "ML_NER_Discovered", 
                    "base_diff": 3 # Default difficulty for newly discovered ML skills
                }

        # --- MERGE & SCORE ---
        reports = []
        is_important = any(sig in jd_lower for sig in self.importance_signals)
        relevance_multiplier = 2.0 if is_important else 1.0

        for key, data in extracted_skills.items():
            skill_name = data["name"]
            
            # 1. Determine Difficulty
            dyn_level = self._get_dynamic_difficulty(jd_text, skill_name)
            difficulty_val = self.difficulty_map.get(dyn_level, data["base_diff"])
            
            # 2. Calculate ERA Priority Score
            p_score = relevance_multiplier * difficulty_val
            
            # 3. Check Resume Match
            is_matched = key in resume_lower
            
            reports.append(SkillReport(
                skill_name=skill_name,
                source=data["source"],
                required_level=dyn_level,
                priority_score=p_score,
                is_matched=is_matched
            ))

        # Sort by Priority Score (Highest first)
        return sorted(reports, key=lambda x: x.priority_score, reverse=True)

### Phase 3: Ingesting Complex Data
Notice in this Job Description, we added **"Snowflake"** and **"Looker"**. 
These do NOT exist in the ERA Dictionary. Let's see if the `ML_NER` engine can catch them!

In [8]:
start_time = time.time()

# JD containing both Dictionary skills (Tableau, Excel) and OOV skills (Snowflake, Looker)
sample_jd = """
Analyze large volumes of client data to inform strategies.
Synthesize complex data analytics into easily understood concepts by creating visualizations.
Develop and optimize dashboards using Tableau and Excel.
Must be an expert in Snowflake data warehousing and proficient in Looker for advanced BI reporting.
"""

sample_resume = """
Data Analyst with 3 years of experience. 
Skilled in creating dashboards using Tableau and Excel. 
Basic knowledge of SQL.
"""

print("✅ Complex Test Data Loaded.")

✅ Complex Test Data Loaded.


###  Phase 4: Execution & Gap Analysis Output
We format the output to clearly show the user what they have, what they are missing, and *how* the system found the skill (showing the power of the dual-engine).

In [9]:
matcher = TrueHybridMatcher(nlp_base, nlp_custom_ner)
results = matcher.process_job_and_resume(jd_text=sample_jd, resume_text=sample_resume)

matched_skills = [r for r in results if r.is_matched]
missing_skills = [r for r in results if not r.is_matched]

print("\n" + "="*85)
print("✅ [MATCHED SKILLS - CANDIDATE HAS THESE]:")
if matched_skills:
    for sk in matched_skills:
        print(f"   - {sk.skill_name:<15} | Level: {sk.required_level:<12} | Score: {sk.priority_score:<4} | ⚙️ Engine: {sk.source}")
else:
    print("   None found.")

print("\n❌ [MISSING SKILLS - RECOMMEND LEARNING SOURCES FOR THESE]:")
if missing_skills:
    for sk in missing_skills:
        print(f"   - {sk.skill_name:<15} | Target Level: {sk.required_level:<5} | Score: {sk.priority_score:<4} | ⚙️ Engine: {sk.source}")
else:
    print("   Perfect Match!")
print("="*85)


✅ [MATCHED SKILLS - CANDIDATE HAS THESE]:
   - Tableau         | Level: Intermediate | Score: 6.0  | ⚙️ Engine: ERA_Dict
   - Excel           | Level: Intermediate | Score: 6.0  | ⚙️ Engine: ERA_Dict

❌ [MISSING SKILLS - RECOMMEND LEARNING SOURCES FOR THESE]:
   - Reporting       | Target Level: Expert | Score: 10.0 | ⚙️ Engine: ERA_Dict
   - Snowflake       | Target Level: Expert | Score: 10.0 | ⚙️ Engine: ML_NER_Discovered
   - Looker          | Target Level: Expert | Score: 10.0 | ⚙️ Engine: ML_NER_Discovered
   - Bi              | Target Level: Expert | Score: 10.0 | ⚙️ Engine: ML_NER_Discovered
   - Data Analytics  | Target Level: Intermediate | Score: 6.0  | ⚙️ Engine: ERA_Dict
   - Synthesize      | Target Level: Intermediate | Score: 6.0  | ⚙️ Engine: ML_NER_Discovered
   - Develop         | Target Level: Intermediate | Score: 6.0  | ⚙️ Engine: ML_NER_Discovered


### 📊 Phase 5: Model Evaluation Metrics (F1-Score Pipeline)
To ensure our Hybrid Model is accurate, we evaluate its extraction capability against a simulated human-annotated `Ground Truth`. We calculate **Precision**, **Recall**, and **F1-Score**.

In [10]:
# Ground Truth: What a human expert says is actually required in the JD
ground_truth_skills: Set[str] = {"Tableau", "Excel", "Data Analytics", "Snowflake", "Looker"}

# What our Dual-Engine extracted
extracted_skills: Set[str] = {r.skill_name for r in results}

# Calculate Confusion Matrix Elements
true_positives = len(ground_truth_skills.intersection(extracted_skills))
false_positives = len(extracted_skills - ground_truth_skills)
false_negatives = len(ground_truth_skills - extracted_skills)

# Metrics Calculation
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"📊 Dual-Engine Evaluation Metrics:")
print(f"Ground Truth : {ground_truth_skills}")
print(f"Extracted    : {extracted_skills}")
print("-" * 45)
print(f"Precision : {precision:.2f} (Accuracy of the extractions)")
print(f"Recall    : {recall:.2f} (Ability to find OOV skills missed by dict)")
print(f"F1-Score  : {f1_score:.2f} (Harmonic Mean)")

end_time = time.time()
print(f"\n⏱️ Execution time: {end_time - start_time:.4f} seconds")

📊 Dual-Engine Evaluation Metrics:
Ground Truth : {'Data Analytics', 'Excel', 'Looker', 'Snowflake', 'Tableau'}
Extracted    : {'Data Analytics', 'Reporting', 'Looker', 'Synthesize', 'Snowflake', 'Develop', 'Tableau', 'Excel', 'Bi'}
---------------------------------------------
Precision : 0.56 (Accuracy of the extractions)
Recall    : 1.00 (Ability to find OOV skills missed by dict)
F1-Score  : 0.71 (Harmonic Mean)

⏱️ Execution time: 0.1681 seconds
